In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

STORAGE = "stecomlakedev"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"
GOLD = f"abfss://gold@{STORAGE}.dfs.core.windows.net"
CKPT = f"abfss://checkpoints@{STORAGE}.dfs.core.windows.net"

# COMMAND ----------
items = spark.read.format("delta").load(f"{SILVER}/order_items")
orders = spark.read.format("delta").load(f"{SILVER}/orders")
customers = spark.read.format("delta").load(f"{SILVER}/customers")
dim_cust = spark.read.format("delta").load(f"{GOLD}/dim_customer").filter("is_current")
dim_prod = spark.read.format("delta").load(f"{GOLD}/dim_product")

# orders.customer_id is per-order; dim_customer is keyed on customer_unique_id,
# so the join has to route through silver customers.
cust_map = customers.select("customer_id", "customer_unique_id")

fact_orders = (items
    .join(orders.select("order_id", "customer_id", "order_status",
                        "order_purchase_timestamp"), "order_id")
    .join(cust_map, "customer_id", "left")
    .join(dim_cust.select(F.col("customer_id").alias("customer_unique_id"),
                          "customer_sk"), "customer_unique_id", "left")
    .join(dim_prod.select("product_id"), "product_id", "left")
    .select(
        "order_id",
        F.col("order_item_id"),
        F.coalesce(F.col("customer_sk"), F.lit(-1)).alias("customer_sk"),
        "product_id",
        "seller_id",
        F.col("price"),
        F.col("freight_value"),
        F.col("item_total"),
        F.to_date("order_purchase_timestamp").alias("order_date"),
        F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int").alias("date_key"),
        "order_status"))

(fact_orders.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_date")
    .save(f"{GOLD}/fact_orders"))

print(f"fact_orders: {spark.read.format('delta').load(f'{GOLD}/fact_orders').count()} rows")

# COMMAND ----------
orphan_sk = spark.read.format("delta").load(f"{GOLD}/fact_orders") \
    .filter("customer_sk IS NULL").count()
print(f"rows with no customer_sk: {orphan_sk}")
assert orphan_sk == 0, "fact rows failed to resolve a customer surrogate key"

# COMMAND ----------
events = spark.readStream.format("delta").load(f"{SILVER}/clickstream_events")

hourly = (events
    .withWatermark("event_ts", "2 hours")
    .groupBy(F.window("event_ts", "1 hour"), "event_type")
    .agg(F.count("*").alias("event_count"),
         F.approx_count_distinct("session_id").alias("sessions"))
    .select(F.col("window.start").alias("hour_start"),
            "event_type", "event_count", "sessions"))

q = (hourly.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT}/gold_hourly")
    .trigger(availableNow=True)
    .start(f"{GOLD}/fact_events_hourly"))
q.awaitTermination()

print(f"fact_events_hourly: {spark.read.format('delta').load(f'{GOLD}/fact_events_hourly').count()} rows")

# COMMAND ----------
display(spark.read.format("delta").load(f"{GOLD}/fact_events_hourly")
        .orderBy("hour_start", "event_type"))